In [5]:
import pandas as pd
import re

file_path = '../data/recipes.csv'
print(f"Loading dataset from {file_path}...")
df = pd.read_csv(file_path)


columns_to_drop = [
    'AuthorId', 'AuthorName', 'ReviewCount', 'DatePublished',
    'RecipeIngredientQuantities', 'RecipeServings', 'RecipeYield'
]
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

print(f"Dataset shape after dropping columns: {df.shape}")

def clean_r_array_string(text):
    if pd.isna(text):
        return []
    text = str(text).strip()
    if text.startswith('c(') and text.endswith(')'):
        text = text[2:-1]
    items = re.findall(r'"([^"]*)"', text)
    items = [item for item in items if item.strip() and item != 'NA']
    if not items and text:
        items = [i.strip() for i in text.split(',') if i.strip() and i.strip() != 'NA']
    return items

array_columns = ['Images', 'RecipeIngredientParts', 'RecipeInstructions', 'Keywords']

print("Cleaning array-like string columns...")
for col in array_columns:
    if col in df.columns:
        df[col] = df[col].apply(clean_r_array_string)

display(df[['RecipeId', 'Name', 'Images', 'RecipeIngredientParts']].head(3))


Loading dataset from ../data/recipes.csv...
Dataset shape after dropping columns: (522517, 21)
Cleaning array-like string columns...


,RecipeId,Name,Images,RecipeIngredientParts
0,38,Low-Fat Berry Blue Frozen Dessert,[https://img.sndimg.com/food/image/upload/w_55...,"[blueberries, granulated sugar, vanilla yogurt..."
1,39,Biryani,[https://img.sndimg.com/food/image/upload/w_55...,"[saffron, milk, hot green chili peppers, onion..."
2,40,Best Lemonade,[https://img.sndimg.com/food/image/upload/w_55...,"[sugar, lemons, rind of, lemon, zest of, fresh..."


In [8]:

df.head()

,RecipeId,Name,CookTime,PrepTime,TotalTime,Description,Images,RecipeCategory,Keywords,RecipeIngredientParts,...,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,PT24H,PT45M,PT24H45M,Make and share this Low-Fat Berry Blue Frozen ...,[https://img.sndimg.com/food/image/upload/w_55...,Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Health...","[blueberries, granulated sugar, vanilla yogurt...",...,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2,"[Toss 2 cups berries with sugar., Let stand fo..."
1,39,Biryani,PT25M,PT4H,PT4H25M,Make and share this Biryani recipe from Food.com.,[https://img.sndimg.com/food/image/upload/w_55...,Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, ...","[saffron, milk, hot green chili peppers, onion...",...,1110.7,58.8,16.6,372.8,368.4,84.4,9.0,20.4,63.4,[Soak saffron in warm milk for 5 minutes and p...
2,40,Best Lemonade,PT5M,PT30M,PT35M,This is from one of my first Good House Keepi...,[https://img.sndimg.com/food/image/upload/w_55...,Beverages,"[Low Protein, Low Cholesterol, Healthy, Summer...","[sugar, lemons, rind of, lemon, zest of, fresh...",...,311.1,0.2,0.0,0.0,1.8,81.5,0.4,77.2,0.3,"[Into a 1 quart Jar with tight fitting lid, pu..."
3,41,Carina's Tofu-Vegetable Kebabs,PT20M,PT24H,PT24H20M,This dish is best prepared a day in advance to...,[https://img.sndimg.com/food/image/upload/w_55...,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[extra firm tofu, eggplant, zucchini, mushroom...",...,536.1,24.0,3.8,0.0,1558.6,64.2,17.3,32.1,29.3,"[Drain the tofu, carefully squeezing out exces..."
4,42,Cabbage Soup,PT30M,PT20M,PT50M,Make and share this Cabbage Soup recipe from F...,[https://img.sndimg.com/food/image/upload/w_55...,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[plain tomato juice, cabbage, onion, carrots, ...",...,103.6,0.4,0.1,0.0,959.3,25.1,4.8,17.7,4.3,"[Mix everything together and bring to a boil.,..."


In [9]:
df.columns

Index(['RecipeId', 'Name', 'CookTime', 'PrepTime', 'TotalTime', 'Description',
       'Images', 'RecipeCategory', 'Keywords', 'RecipeIngredientParts',
       'AggregatedRating', 'Calories', 'FatContent', 'SaturatedFatContent',
       'CholesterolContent', 'SodiumContent', 'CarbohydrateContent',
       'FiberContent', 'SugarContent', 'ProteinContent', 'RecipeInstructions'],
      dtype='object')

In [ ]:
import asyncio
import aiohttp
import nest_asyncio
import pandas as pd
import re
from rapidfuzz import process
from tqdm.asyncio import tqdm

nest_asyncio.apply()

print("--- Step 1: Extract First URL ---")
def get_first_url(img_str):
    if pd.isna(img_str) or img_str == "character(0)":
        return None
    urls = re.findall(r'"(https?://[^"]+)"', str(img_str))
    return urls[0] if urls else None

df['temp_url'] = df['Images'].apply(get_first_url)

print("\n--- Step 2: Async Fast URL Checker ---")
async def check_url_status(session, idx, url):
    if not url:
        return idx, url, False
    try:
        async with session.head(url, timeout=5, allow_redirects=True) as response:
            return idx, url, response.status == 200
    except:
        return idx, url, False

async def validate_urls(df_to_check, limit=100):
    connector = aiohttp.TCPConnector(limit=limit)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [check_url_status(session, idx, row['temp_url'])
                 for idx, row in df_to_check.dropna(subset=['temp_url']).iterrows()]
        return await tqdm.gather(*tasks, desc="Checking URLs")

validation_results = asyncio.run(validate_urls(df, limit=100))

valid_indices = [res[0] for res in validation_results if res[2] is True]
df['verified_url'] = None
df.loc[valid_indices, 'verified_url'] = df.loc[valid_indices, 'temp_url']

print(f"Alive URLs: {len(valid_indices):,}")
print(f"Dead/Missing URLs: {len(df) - len(valid_indices):,}")

print("\n--- Step 3: RapidFuzz Imputation ---")
valid_df = df.dropna(subset=['verified_url']).drop_duplicates(subset=['Name'])
image_lookup_table = dict(zip(valid_df['Name'].str.lower(), valid_df['verified_url']))
unique_missing_names = df[df['verified_url'].isna()]['Name'].dropna().str.lower().unique()

fallback_map = {}
for name in unique_missing_names:
    match = process.extractOne(name, image_lookup_table.keys(), score_cutoff=85)
    fallback_map[name] = image_lookup_table[match[0]] if match else None

df['fallback_image'] = df['Name'].str.lower().map(fallback_map)

print("\n--- Step 4: Finalizing and Saving ---")
df['Images'] = df['verified_url'].fillna(df['fallback_image'])

df_final = df.drop(columns=['temp_url', 'verified_url', 'fallback_image'], errors='ignore')
final_pkl = 'recipes_ready_for_es.pkl'
df_final.to_pickle(final_pkl)
print(f"Success !!!: {final_pkl}")